# Scenario 6: Structured Data Extraction Pipeline
## Hands-on Production Scenario Workbench
**Certification Focus**: CCAR-F Exam Scenarios 1–6 | **Standard**: Claude 4.5 & Extended Thinking

---

### 📊 Production SLAs & Non-Negotiables
- Tool Use with JSON Schema as gold standard for deterministic schema conformity
- Explicit nullable fields to prevent fabrication on missing facts
- Extensible taxonomy (enum with 'other' + other_description string)
- Closed-loop validation: early termination on missing facts; targeted retry on math errors
- Calibrated tiered routing: Human triage for low confidence (<0.70) + Stratified audit

---

### 🏗️ Architecture & Decision Flow
```mermaid
flowchart TD
    DocInput["Raw Invoice Document"] --> Extract["Tool Use Extraction<br>(tool_choice: forced)"]
    
    Extract --> Validator{"Closed-Loop Validator"}
    
    Validator -->|"Missing Source Fact"| AbortMissing["Halt Retries Immediately<br>(Mark Nullable / Escalate)"]
    Validator -->|"Arithmetic Discrepancy"| RetryFeedback["Targeted Reprompt with Error<br>(Max 2 retries)"]
    Validator -->|"Schema Valid"| ConfidenceCheck{"Calibrated Confidence"}
    
    ConfidenceCheck -->|"Confidence < 0.70"| HumanTriage["Mandatory Human Triage"]
    ConfidenceCheck -->|"Confidence >= 0.90"| StratifiedCheck{"Stratified 5% Audit?"}
    StratifiedCheck -->|"Sampled"| QAQueue["QA Review Queue"]
    StratifiedCheck -->|"Not Sampled"| StraightThrough["Automated Database Commit"]
```

---

### 🏆 Golden Architectural Rules Reference
| Rule | Production Implementation & Rationale |
|---|---|
| **Tool Use with JSON Schema (TS 4.3)** | Use tool calling to guarantee 100% compliant JSON structures without regex. |
| **Explicit Nullable Types (TS 4.3)** | Allow null for optional fields to prevent hallucinated values on absent facts. |
| **Stop on Unfixable Errors (TS 4.4)** | When a required fact is genuinely missing from source, abort retry loop immediately. |
| **Targeted Feedback on Retry (TS 4.4)** | Provide specific validation error details on recoverable mathematical mismatches. |
| **Stratified Random Sampling (TS 5.5)** | Audit 5-10% of high-confidence predictions across all document classes to detect drift. |

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns.
- **Case Facts Preservation**: Immutable transactional entities must be maintained in a dedicated Case Facts block outside conversation summaries.



In [ ]:
# Step 1: Environment Setup & Simulation Import
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scenarios.scenario_6_structured_data_extraction.simulation import *

print(' Scenario Workbench Initialized: Scenario 6: Structured Data Extraction Pipeline')
print(' Zero live API calls, 100% offline determinism')


### 🧪 Step 2: Core Execution Flow & Happy Path Simulation
Demonstrating end-to-end pipeline execution with audit logging.


In [ ]:
# Test Math Error Recovery via Retry
pipeline = StructuredExtractionPipeline(random_seed=42)
doc_text = "Subtotal: $50, Tax: $5, Total: $55, Tax ID: TX-1"
turns = [
    {"subtotal": 50.0, "tax": 5.0, "total": 50.0, "tax_id": "TX-1"},
    {"subtotal": 50.0, "tax": 5.0, "total": 55.0, "tax_id": "TX-1"}
]
outcome = pipeline.process_extraction("d1", doc_text, turns, confidence_score=0.96)
print("Extraction Status:", outcome.status)
print("Retries Used:", outcome.retries_used)
print("Routing Tier:", outcome.routing_tier)
assert outcome.status == "SUCCESS"
assert outcome.retries_used == 1



### ⚠️ Step 3: Edge Case Testing & Guardrail Verifications
Testing boundary conditions, safety gates, error recovery, and escalation protocols.


In [ ]:
# Test Unrecoverable Missing Fact Aborts Retry Loop Immediately
pipeline = StructuredExtractionPipeline(random_seed=42)
doc_missing = "Simple receipt: Total $10. Paid in cash at diner counter."
turns_missing = [{"subtotal": 10.0, "tax": 0.0, "total": 10.0, "tax_id": None}]
outcome_missing = pipeline.process_extraction("d2", doc_missing, turns_missing, confidence_score=0.95)
print("Missing Fact Status:", outcome_missing.status)
print("Retries Used:", outcome_missing.retries_used)
assert outcome_missing.status == "FAILED_UNRECOVERABLE"
assert outcome_missing.retries_used == 0
assert outcome_missing.routing_tier == ExtractionRoutingTier.HUMAN_TRIAGE



### ✅ Step 4: Verification Summary & SLA Compliance
All assertions passed with zero unauthorized actions and 100% deterministic compliance.
